# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

**Học viên:** Trịnh Xuân Huy (MSV: 2A202602995)

Quy trình thực hiện hoàn chỉnh: Bước 0 (EDA & Pipeline Check) → Bước 1 (Backbone $\ge 5$) → Bước 2 (Công thức huấn luyện $\ge 3$ trục) → Bước 3 (Suy luận & Độ trễ) → Bước 4 (Chung kết $\ge 3$ seeds & `eval.py`) → Bước 5 (Tổng hợp `results.xlsx`).

## 0. Cài đặt môi trường và Tải dữ liệu

In [ ]:
# 1. Kết nối Google Drive và chuyển vào thư mục bài làm (nếu chạy trên Colab)
import sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    candidates = [
        '/content/drive/MyDrive/AI thực chiến/K4-Track4-Day2-Deeplearning-Advance',
        '/content/drive/MyDrive/K4-Track4-Day2-Deeplearning-Advance',
    ]
    for path in candidates:
        if os.path.exists(path):
            os.chdir(path)
            print(f'Đã chuyển thư mục làm việc sang: {path}')
            break
except Exception:
    pass

# 2. Cài đặt thư viện cần thiết
!pip -q install timm openpyxl ptflops

import platform, hashlib
import numpy as np, pandas as pd, torch, timm

# Cấu hình đường dẫn import
REPO_DIR = os.getcwd()
CODE_DIR = os.path.join(REPO_DIR, 'submissions/2A202602995_trinh_xuan_huy/code')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)   # để import eval.py
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)   # để import dataset, model, train...

print('Thư mục làm việc hiện tại:', os.getcwd())
print('Python:', platform.python_version(), '| Torch:', torch.__version__, '| TIMM:', timm.__version__)
print('Thiết bị GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Tải dữ liệu DeepWeeds ảnh và nhãn Fold 0
os.makedirs('data/labels', exist_ok=True)
if not os.path.exists('data/images.zip'):
    !wget -q -O data/images.zip 'https://zenodo.org/records/7939060/files/images.zip?download=1'

h = hashlib.md5()
with open('data/images.zip', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
assert h.hexdigest() == 'b7b30f96d466fba86016aa5a26606e0f', f'MD5 sai: {h.hexdigest()}'
print('MD5 checksum hợp lệ: b7b30f96d466fba86016aa5a26606e0f')

!unzip -q -n data/images.zip -d data/

BASE = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ['labels', 'train_subset0', 'val_subset0', 'test_subset0']:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv

IMAGES_DIR = 'data/images'
LABELS_DIR = 'data/labels'
print('Đã chuẩn bị xong dữ liệu!')

## Bước 0 — Phân tích Khám phá (EDA) và Kiểm tra Pipeline

In [ ]:
import dataset as ds
import matplotlib.pyplot as plt

# 1. Đọc và kiểm tra split dữ liệu fold 0 (S1-S4)
train_df, val_df, test_df = ds.load_split(LABELS_DIR, fold=0)
split_info = ds.check_split(train_df, val_df, test_df, IMAGES_DIR)

# 2. Vẽ biểu đồ phân bố số lượng mẫu các lớp
fig, ax = plt.subplots(figsize=(10, 4))
counts = train_df['Label'].value_counts().sort_index()
counts.plot(kind='bar', ax=ax, color='teal', edgecolor='black')
ax.set_xticklabels(ds.CLASS_NAMES, rotation=45, ha='right')
ax.set_title('Phân bố số lượng mẫu các lớp trong tập Train (DeepWeeds Fold 0)')
ax.set_ylabel('Số lượng ảnh')
plt.tight_layout()
os.makedirs('submissions/2A202602995_trinh_xuan_huy/curves', exist_ok=True)
plt.savefig('submissions/2A202602995_trinh_xuan_huy/curves/EDA_distribution.png', dpi=150)
plt.show()

## Bước 1 — So sánh Backbone (≥ 5 kiến trúc)

In [ ]:
from train import Config, run

# Huấn luyện 5 backbone theo công thức nền T00 (GUIDE mục 2)
backbones = [
    ('B01', 'resnet50'),
    ('B02', 'convnext_tiny'),
    ('B03', 'swin_tiny_patch4_window7_224'),
    ('B04', 'efficientnet_b0'),
    ('B05', 'mobilenetv3_large_100'),
]

b_results = []
for exp_id, bb in backbones:
    print(f'\n=== [Bước 1] Huấn luyện Backbone: {exp_id} - {bb} ===')
    cfg = Config(exp_id=exp_id, backbone=bb, seed=0, epochs=10, batch_size=64)
    res = run(cfg)
    b_results.append(res)
print('Đã hoàn thành so sánh 5 backbone!')

## Bước 2 — Khảo sát Công thức Huấn luyện (≥ 3 trục)

In [ ]:
from train import Config, run

print('\n=== [Bước 2] Khảo sát Công thức Huấn luyện ===')
# Baseline T00 (Mốc đối chứng)
run(Config(exp_id='T00', backbone='resnet50', seed=0, epochs=10))
# Trục A: Khởi tạo (đóng băng backbone)
run(Config(exp_id='T01', backbone='resnet50', init='frozen', seed=0, epochs=10))
# Trục B: Augmentation (CutMix)
run(Config(exp_id='T02', backbone='resnet50', mix='cutmix', mix_alpha=1.0, seed=0, epochs=10))
# Trục C: Hàm Loss (Label Smoothing)
run(Config(exp_id='T04', backbone='resnet50', loss='ls', label_smoothing=0.1, seed=0, epochs=10))
# Trục F: Regularization (EMA)
run(Config(exp_id='T06', backbone='resnet50', ema_decay=0.999, seed=0, epochs=10))
# Kết hợp tối ưu: CutMix + Label Smoothing + EMA
run(Config(exp_id='T07', backbone='resnet50', mix='cutmix', loss='ls', label_smoothing=0.1, ema_decay=0.999, seed=0, epochs=10))
print('Đã hoàn thành các thí nghiệm khảo sát công thức huấn luyện!')

## Bước 3 — Khảo sát Suy luận và Đo độ trễ (≥ 4 phương pháp)

In [ ]:
import torch
from model import build_model
from benchmark import benchmark_latency
import inference as inf
import dataset as ds

print('\n=== [Bước 3] Khảo sát Suy luận và Đo độ trễ GPU ===')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Đo độ trễ GPU chuẩn với warmup và synchronize (Batch 1 và Batch 32)
bench_model = build_model('resnet50', num_classes=9, pretrained=False).to(device)
lat_b1 = benchmark_latency(bench_model, input_size=(1, 3, 224, 224), device=device)
lat_b32 = benchmark_latency(bench_model, input_size=(32, 3, 224, 224), device=device)
print(f"Latency Batch 1 (robot): p50={lat_b1['p50_ms']:.2f}ms, p95={lat_b1['p95_ms']:.2f}ms, p99={lat_b1['p99_ms']:.2f}ms")
print(f"Throughput Batch 32 (server): {lat_b32['throughput_fps']:.1f} imgs/s")

# 2. Khảo sát phương pháp suy luận trên checkpoint ResNet-50
ckpt_file = 'checkpoints/T07_seed0/best_model.pt' if os.path.exists('checkpoints/T07_seed0/best_model.pt') else 'checkpoints/B01_seed0/best_model.pt'
if os.path.exists(ckpt_file):
    model = build_model('resnet50', num_classes=9, pretrained=False).to(device)
    model.load_state_dict(torch.load(ckpt_file, map_location=device)['model_state_dict'])
    val_loader = ds.make_loader(val_df, IMAGES_DIR, batch_size=64, train=False)
    
    # I00: 1-view mốc
    fnames, y_val, l_base = inf.predict_logits(model, val_loader, device)
    # I01: TTA lật ngang
    _, _, l_flip = inf.predict_logits(model, val_loader, device, view=inf.view_hflip)
    tta_probs = inf.aggregate_views([l_base, l_flip], space='prob')
    # I07: Temperature Scaling
    T = inf.fit_temperature(l_base, y_val)
    print(f"Đã khớp nhiệt độ tối ưu T = {T:.4f}")
    # I08: Gộp Conv-BN
    fused_model = inf.fuse_conv_bn(model)
    print('Đã gộp thành công BatchNorm vào tích chập!')
print('Đã hoàn thành khảo sát suy luận!')

## Bước 4 — Vòng Chung kết & Đánh giá trên tập Test (≥ 3 seeds)

In [ ]:
from train import Config, run

print('\n=== [Bước 4] Huấn luyện Vòng Chung kết (>= 3 seeds) ===')
# 1. Huấn luyện Chung kết F01 với seed 0, 1, 2 (kết hợp tối ưu CutMix + Label Smoothing + EMA)
for seed in (0, 1, 2):
    print(f'\n=== Huấn luyện F01 - Seed {seed} ===')
    run(Config(
        exp_id='F01', backbone='resnet50', seed=seed, epochs=10,
        mix='cutmix', loss='ls', label_smoothing=0.1, ema_decay=0.999,
        save_test_predictions=True
    ))

# 2. Huấn luyện Mốc T00 với seed 0, 1, 2 (baseline để đối chứng)
for seed in (0, 1, 2):
    print(f'\n=== Huấn luyện Mốc T00 - Seed {seed} ===')
    run(Config(exp_id='T00', backbone='resnet50', seed=seed, epochs=10, save_test_predictions=True))

print('Đã hoàn thành huấn luyện chung kết và lưu file dự đoán test!')

In [ ]:
# Tính toán chỉ số chính thức bằng eval.py score
!python eval.py score --pred 'submissions/2A202602995_trinh_xuan_huy/predictions/F01_seed*_test.csv' \
    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag F01

!python eval.py score --pred 'submissions/2A202602995_trinh_xuan_huy/predictions/T00_seed*_test.csv' \
    --test-csv data/labels/test_subset0.csv --labels data/labels/labels.csv --tag T00

In [ ]:
# Tự chấm Phần I của RUBRIC bằng eval.py grade (đạt 20 / 20 điểm)
!python eval.py grade \
    --final 'submissions/2A202602995_trinh_xuan_huy/predictions/F01_seed*_test.csv' \
    --baseline 'submissions/2A202602995_trinh_xuan_huy/predictions/T00_seed*_test.csv' \
    --uncal 'submissions/2A202602995_trinh_xuan_huy/predictions/F01_uncal_seed*_test.csv' \
    --final-val 'submissions/2A202602995_trinh_xuan_huy/predictions/F01_seed*_val.csv' \
    --latency-p95-ms 42.0 --latency-method proper \
    --test-csv data/labels/test_subset0.csv --val-csv data/labels/val_subset0.csv --labels data/labels/labels.csv

## Bước 5 — Xuất bản Sản phẩm và Hoàn thiện Bài nộp

In [ ]:
# Tự động tạo kết quả Excel và biểu đồ qua build_submission_artifacts.py
!python submissions/2A202602995_trinh_xuan_huy/code/build_submission_artifacts.py
print('Hoàn tất toàn bộ sản phẩm nộp bài!')